In [1]:
import sys
import json
from tqdm import tqdm
from openai import OpenAI 
import os
from functools import reduce
import joblib
from typing import Dict
import gc

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.memorize_pipeline import MemPipeline
from src.memorize_pipeline.extractor.LLMExtractor import LLMExtractor
from src.memorize_pipeline.updator.LLMUpdator import LLMUpdator
from src.llm_agent import AgentConnector
from src.utils.data_structs import TripletCreator, NodeCreator, Relation, NODES_TYPES_MAP, RELATIONS_TYPES_MAP
from src.llm_agent.agent_model import SYSTEM_PROMPT

from src.neo4j_functions import Neo4jConnection
from src.embedding_functions import EmbeddingsDatabaseConnection, EmbeddingsDatabaseConnectionConfig, VectorDBConnectionConfig
from src.knowledge_graph_model import KnowledgeGraphModel

NEO4J_URL ="bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PWD = "password"

GRAPH_DB_NAME = 'diaasq2'
gc.collect()

/home/dzigen/Desktop/PersonalAI/Personal-AI/pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


20

### Update

In [2]:
kg_model = KnowledgeGraphModel(
    graph_db=Neo4jConnection(uri=NEO4J_URL, user=NEO4J_USER, pwd=NEO4J_PWD, db_name=GRAPH_DB_NAME),
    embeddings_db=EmbeddingsDatabaseConnection(EmbeddingsDatabaseConnectionConfig(
        nodes_db_config=VectorDBConnectionConfig(
            '../data/graph_structures/vectorized_nodes/v10/densedb', 'vectorized_nodes', is_exist=True, need_to_clear=True
        ),
        triplets_db_config=VectorDBConnectionConfig(
            '../data/graph_structures/vectorized_triplets/v6/densedb', 'vectorized_triplets', is_exist=True, need_to_clear=True
        )
    ))
)

No sentence-transformers model found with name ../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [3]:
raw_triplets = list(kg_model.graph_db.execute_query("MATCH (n1)-[rel]->(n2) RETURN n1, rel, n2"))

formated_triplets = []
for raw_triplet in tqdm(raw_triplets):
    start_node = NodeCreator.create(id=raw_triplet['n1'].element_id, name=raw_triplet['n1']['name'], 
                                    type=NODES_TYPES_MAP[list(raw_triplet['n1'].labels)[0]],
                                    prop=dict(raw_triplet['n1']))
    end_node = NodeCreator.create(id=raw_triplet['n2'].element_id, name=raw_triplet['n2']['name'], 
                                    type=NODES_TYPES_MAP[list(raw_triplet['n2'].labels)[0]],
                                    prop=dict(raw_triplet['n2']))
    relation = Relation(id=raw_triplet['rel'].element_id, name=raw_triplet['rel']['name'], 
                        type=RELATIONS_TYPES_MAP[raw_triplet['rel'].type], 
                        prop=dict(raw_triplet['rel']))
    
    triplet = TripletCreator.create(start_node, relation, end_node, add_stringified_triplet=False)
    formated_triplets.append(triplet)

100%|██████████| 283268/283268 [00:08<00:00, 34854.32it/s]


In [4]:
kg_model.embeddings_db.add_triplets(
    formated_triplets, batch_size=128)

100%|██████████| 2214/2214 [59:14<00:00,  1.61s/it] 

all/unique_triplets - 283268/72280
all/unique_nodes - 566536/71338


In [5]:
kg_model.graph_db.close()